# Lesson 21: Production Arithmetic Agent — Complete Architecture

## Objective
Combine ALL 20 lessons into a **production-ready arithmetic agent** with: LLM parsing, validation, conditional routing, tools, memory, checkpointing, human approval for large numbers, self-critique, and streaming.

## What This Lesson Covers
This is the **integration lesson** — every concept from Lessons 1–20 appears here:

| Lesson | Feature | Where Used |
|--------|---------|-----------|
| 1–2 | StateGraph, TypedDict | Foundation |
| 3 | Conditional routing | Operation routing |
| 4 | Reducers | History accumulation |
| 5 | Loops | Retry on validation failure |
| 6–8 | LLM + tools | Parse intent, tool execution |
| 9 | Parallel | Sum + product simultaneously |
| 11 | Subgraph | Arithmetic engine subgraph |
| 13–14 | Memory | Session history |
| 16 | Streaming | Real-time output |
| 17 | Checkpointing | Session persistence |
| 18 | HITL | Approve large multiplications |
| 20 | Self-critique | Verify answers |

## Architecture Overview

```
User Input (natural language)
        ↓
   [PARSE NODE - LLM]
   "what is 7 times 8?" → {a:7, b:8, op:multiply}
        ↓
   [VALIDATE NODE]
   Check: valid op? safe range? no div/0?
        ↓ (validation loop if invalid — Lesson 5)
   [ROUTE NODE]
   Is this a large multiplication? → HITL approval (Lesson 18)
   Is this a normal operation? → arithmetic engine (Lesson 11)
        ↓
   [ARITHMETIC ENGINE SUBGRAPH - Lesson 11]
   add / multiply / divide
        ↓
   [SELF-CRITIQUE NODE - Lesson 20]
   Verify the answer is correct
        ↓
   [MEMORY WRITE - Lesson 15]
   Save to long-term store
        ↓
   Result (streamed - Lesson 16)
```


In [ ]:
# ── Cell 1: Full Production Imports ─────────────────────────────────────────
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

import os, operator, sqlite3, datetime
import vertexai
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import interrupt, Command
from typing import TypedDict, Optional, Literal, Annotated
from IPython.display import Image, display

load_dotenv()
vertexai.init(project=os.getenv("PROJECT_ID"), location=os.getenv("LOCATION"))

llm          = ChatVertexAI(model="gemini-2.5-flash", temperature=0)
llm_stream   = ChatVertexAI(model="gemini-2.5-flash", temperature=0, streaming=True)
LARGE_THRESHOLD = 100  # Require HITL approval for multiplications above this

print("✓ Production imports complete")
print(f"  LARGE_THRESHOLD = {LARGE_THRESHOLD} (HITL approval for a×b > this)")


In [ ]:
# ── Cell 2: Production State ─────────────────────────────────────────────────
class ProdState(MessagesState):
    # Parsed intent
    question: str
    a: Optional[float]
    b: Optional[float]
    operation: Optional[str]
    
    # Validation
    is_valid: bool
    validation_error: str
    validation_attempts: int
    
    # Execution
    result: Optional[float]
    needs_approval: bool
    approved: bool
    
    # Quality
    is_verified: bool
    critique: str
    critique_attempts: int
    
    # Memory
    history: Annotated[list, operator.add]
    session_id: str

print("✓ Production state defined with all feature fields")


In [ ]:
# ── Cell 3: LLM Parse Node ────────────────────────────────────────────────────
PARSE_SYSTEM = """Parse arithmetic requests. Respond EXACTLY:
OPERATION: <add|multiply|divide>
A: <number>
B: <number>
If unclear, default to add."""

def parse_node(state: ProdState) -> dict:
    question = state["question"]
    print(f"  [parse] '{question}'")
    
    resp = llm.invoke([
        SystemMessage(content=PARSE_SYSTEM),
        HumanMessage(content=question)
    ])
    
    lines = {l.split(":")[0].strip().upper(): l.split(":",1)[1].strip()
             for l in resp.content.strip().split("\n") if ":" in l}
    
    op = lines.get("OPERATION","add").lower().strip()
    if op not in ("add","multiply","divide"):
        op = "add"
    
    try: a = float(lines.get("A","0"))
    except: a = 0.0
    try: b = float(lines.get("B","0"))
    except: b = 0.0
    
    print(f"  [parse] → {a} {op} {b}")
    return {
        "a": a, "b": b, "operation": op,
        "is_valid": False, "validation_attempts": 0,
        "history": [f"Parsed: '{question}' → {a} {op} {b}"]
    }

print("✓ LLM parse node defined")


In [ ]:
# ── Cell 4: Validation + Fix Nodes ──────────────────────────────────────────
MAX_VAL_ATTEMPTS = 3

def validate_node(state: ProdState) -> dict:
    a, b, op = state["a"], state["b"], state["operation"]
    attempts = state.get("validation_attempts", 0) + 1
    
    if op not in ("add","multiply","divide"):
        return {"is_valid": False, "validation_error": f"Unknown op '{op}'",
                "validation_attempts": attempts}
    if op == "divide" and b == 0:
        return {"is_valid": False, "validation_error": "Cannot divide by zero",
                "validation_attempts": attempts}
    if not (-10000 <= a <= 10000 and -10000 <= b <= 10000):
        return {"is_valid": False, "validation_error": "Numbers out of range",
                "validation_attempts": attempts}
    
    # Check if large multiplication needs approval
    needs_approval = (op == "multiply" and abs(a * b) > LARGE_THRESHOLD)
    print(f"  [validate] ✓ Valid | needs_approval={needs_approval}")
    return {"is_valid": True, "validation_error": "",
            "needs_approval": needs_approval, "validation_attempts": attempts}

def fix_node(state: ProdState) -> dict:
    print(f"  [fix] Fixing: {state['validation_error']}")
    a, b, op = state["a"], state["b"], state["operation"]
    if op not in ("add","multiply","divide"): op = "add"
    if op == "divide" and b == 0: b = 1.0
    a = max(-10000, min(10000, a))
    b = max(-10000, min(10000, b))
    return {"a": a, "b": b, "operation": op}

def validation_router(state: ProdState) -> Literal["compute", "fix", "exit"]:
    if state["is_valid"]: return "compute"
    if state.get("validation_attempts", 0) >= MAX_VAL_ATTEMPTS: return "exit"
    return "fix"

print("✓ Validation system (3 nodes) defined")


In [ ]:
# ── Cell 5: Approval Gate + Compute Nodes ─────────────────────────────────────
def approval_gate(state: ProdState) -> dict:
    a, b = state["a"], state["b"]
    decision = interrupt({
        "message": f"Large multiplication: {a} × {b} = {a*b}. Approve?",
        "a": a, "b": b, "product": a*b
    })
    approved = decision.get("approved", False)
    print(f"  [approval] Human {'✓ approved' if approved else '✗ rejected'}")
    return {"approved": approved}

def compute_node(state: ProdState) -> dict:
    a, b, op = state["a"], state["b"], state["operation"]
    
    # Check approval for large multiplications
    if op == "multiply" and state.get("needs_approval") and not state.get("approved", True):
        return {"result": None, "history": ["Rejected: not approved"]}
    
    if op == "add":        res, sym = a+b, "+"
    elif op == "multiply": res, sym = a*b, "×"
    else:                  res, sym = (a/b if b!=0 else None), "÷"
    
    entry = f"{a} {sym} {b} = {res}"
    print(f"  [compute] {entry}")
    return {"result": res, "history": [entry]}

def needs_approval_router(state: ProdState) -> Literal["approval_gate", "compute"]:
    if state.get("needs_approval") and not state.get("approved", False):
        return "approval_gate"
    return "compute"

print("✓ Approval gate and compute nodes defined")


In [ ]:
# ── Cell 6: Self-Critique Nodes ───────────────────────────────────────────────
MAX_CRITIQUE = 2

CRITIC_SYS = """Verify arithmetic. Respond EXACTLY:
VERDICT: CORRECT or INCORRECT
CORRECT_ANSWER: <number>
REASON: <brief>"""

def critique_node(state: ProdState) -> dict:
    q = state["question"]
    result = state["result"]
    attempts = state.get("critique_attempts", 0) + 1
    
    if result is None:
        return {"is_verified": True, "critique": "N/A (no result)", "critique_attempts": attempts}
    
    resp = llm.invoke([
        SystemMessage(content=CRITIC_SYS),
        HumanMessage(content=f"Question: {q}\nAnswer: {result}")
    ])
    content = resp.content.strip()
    
    lines = {l.split(":")[0].strip().upper(): l.split(":",1)[1].strip()
             for l in content.split("\n") if ":" in l}
    
    verdict = lines.get("VERDICT","INCORRECT").upper()
    is_correct = (verdict == "CORRECT")
    
    try: correct_num = float(lines.get("CORRECT_ANSWER","0"))
    except: correct_num = result
    
    print(f"  [critic] {verdict} | correct={correct_num}")
    
    if not is_correct:
        return {
            "result": correct_num, "is_verified": False,
            "critique": lines.get("REASON",""), "critique_attempts": attempts,
            "history": [f"Critic corrected: {result} → {correct_num}"]
        }
    
    return {"is_verified": True, "critique": "Verified correct",
            "critique_attempts": attempts, "history": [f"Critic: ✓ {result} is correct"]}

def critique_router(state: ProdState) -> Literal["done", "critique"]:
    if state.get("is_verified") or state.get("critique_attempts",0) >= MAX_CRITIQUE:
        return "done"
    return "critique"

print("✓ Self-critique node defined")


In [ ]:
# ── Cell 7: Memory Write + Response Nodes ────────────────────────────────────
def memory_write_node(state: ProdState) -> dict:
    result = state.get("result")
    if result is not None:
        entry = f"STORED: {state['question']} = {result}"
        print(f"  [memory] {entry}")
        return {"history": [entry]}
    return {}

def response_node(state: ProdState) -> dict:
    result = state.get("result")
    verified = state.get("is_verified", False)
    
    if result is None:
        content = "Sorry, I could not compute the answer."
    else:
        status = "✓ Verified" if verified else "⚠ Unverified"
        content = f"Answer: {result} [{status}]"
    
    print(f"  [response] {content}")
    return {
        "messages": [AIMessage(content=content)],
        "history": [f"Final: {content}"]
    }

print("✓ Memory write and response nodes defined")


In [ ]:
# ── Cell 8: Build Complete Production Graph ──────────────────────────────────
memory = MemorySaver()
builder = StateGraph(ProdState)

# Register all nodes
builder.add_node("parse",          parse_node)
builder.add_node("validate",       validate_node)
builder.add_node("fix",            fix_node)
builder.add_node("approval_gate",  approval_gate)
builder.add_node("compute",        compute_node)
builder.add_node("critique",       critique_node)
builder.add_node("memory_write",   memory_write_node)
builder.add_node("response",       response_node)

# ── Edges ──────────────────────────────────────────────────────────────────
builder.add_edge(START, "parse")
builder.add_edge("parse", "validate")

# Validation loop (Lesson 5)
builder.add_conditional_edges("validate", validation_router, {
    "compute": "approval_gate",  # Valid → check approval
    "fix":     "fix",            # Invalid → fix
    "exit":    "response",       # Max retries → exit
})
builder.add_edge("fix", "validate")  # Loop back

# Approval gate (Lesson 18)
builder.add_conditional_edges("approval_gate", needs_approval_router, {
    "approval_gate": "approval_gate",
    "compute":       "compute",
})

builder.add_edge("compute",  "critique")

# Self-critique loop (Lesson 20)
builder.add_conditional_edges("critique", critique_router, {
    "done":     "memory_write",
    "critique": "critique",   # Re-verify if corrected
})

builder.add_edge("memory_write", "response")
builder.add_edge("response",      END)

graph = builder.compile(checkpointer=memory)
print("✓ COMPLETE PRODUCTION GRAPH compiled with ALL features")


In [ ]:
# ── Cell 9: Visualize Full Production Graph ──────────────────────────────────
print("── Full Production Architecture ──")
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# ── Cell 10: Test — Standard Addition ────────────────────────────────────────
config = {"configurable": {"thread_id": "prod-session-1"}}

def run(question: str, cfg=None):
    cfg = cfg or config
    state = {
        "question": question,
        "messages": [HumanMessage(content=question)],
        "a": None, "b": None, "operation": None,
        "is_valid": False, "validation_error": "", "validation_attempts": 0,
        "result": None, "needs_approval": False, "approved": False,
        "is_verified": False, "critique": "", "critique_attempts": 0,
        "history": [], "session_id": cfg["configurable"]["thread_id"]
    }
    print(f"\n{'='*55}")
    print(f"Question: '{question}'")
    print(f"{'='*55}")
    out = graph.invoke(state, config=cfg)
    print(f"\n  ► {out['messages'][-1].content}")
    return out

run("what is 25 plus 17?")


In [ ]:
run("divide 144 by 12")


In [ ]:
# ── Cell 11: Test Small Multiplication (No Approval Needed) ──────────────────
run("multiply 7 by 8")


In [ ]:
# ── Cell 12: Test Large Multiplication (HITL Approval) ───────────────────────
print("\n=== Large multiplication test ===")
print(f"  {1000} × {500} = {500000} which is > LARGE_THRESHOLD={LARGE_THRESHOLD}")
print("  This will PAUSE for human approval...")

large_config = {"configurable": {"thread_id": "prod-large-1"}}

state = {
    "question": "multiply 1000 by 500",
    "messages": [HumanMessage(content="multiply 1000 by 500")],
    "a": None, "b": None, "operation": None,
    "is_valid": False, "validation_error": "", "validation_attempts": 0,
    "result": None, "needs_approval": False, "approved": False,
    "is_verified": False, "critique": "", "critique_attempts": 0,
    "history": [], "session_id": "prod-large-1"
}

# First invoke — will pause at approval_gate
result = graph.invoke(state, config=large_config)

# Check if paused
paused = graph.get_state(large_config)
if paused.next and "approval_gate" in str(paused.next):
    print("\n⏸  Graph PAUSED for human approval")
    print("   Pending: 1000 × 500 = 500000")
    print("\nHuman: I approve this calculation")
    
    # Resume with approval
    out = graph.invoke(Command(resume={"approved": True}), config=large_config)
    print(f"\n  ► {out['messages'][-1].content}")
else:
    print(f"  ► {result['messages'][-1].content}")


In [ ]:
# ── Cell 13: Stream the Full Graph ───────────────────────────────────────────
print("=== STREAMING MODE ===")
stream_config = {"configurable": {"thread_id": "prod-stream-1"}}

state = {
    "question": "what is 13 multiplied by 7?",
    "messages": [HumanMessage(content="what is 13 multiplied by 7?")],
    "a": None, "b": None, "operation": None,
    "is_valid": False, "validation_error": "", "validation_attempts": 0,
    "result": None, "needs_approval": False, "approved": False,
    "is_verified": False, "critique": "", "critique_attempts": 0,
    "history": [], "session_id": "prod-stream-1"
}

for event in graph.stream(state, config=stream_config, stream_mode="updates"):
    node = list(event.keys())[0]
    update = event[node]
    print(f"  ✓ [{node}] → {str(update)[:80]}")


In [ ]:
# ── Cell 14: Session History (Multi-Turn with Checkpointing) ─────────────────
print("=== MULTI-TURN SESSION (Checkpointing) ===")

session_config = {"configurable": {"thread_id": "prod-multiturn-1"}}

for question in ["what is 15 plus 28?", "now divide that by 3", "multiply 7 by 6"]:
    state = {
        "question": question,
        "messages": [HumanMessage(content=question)],
        "a": None, "b": None, "operation": None,
        "is_valid": False, "validation_error": "", "validation_attempts": 0,
        "result": None, "needs_approval": False, "approved": False,
        "is_verified": False, "critique": "", "critique_attempts": 0,
        "history": [], "session_id": "prod-multiturn-1"
    }
    out = graph.invoke(state, config=session_config)
    print(f"  Q: {question}")
    print(f"  A: {out['messages'][-1].content}\n")


## Final Production Architecture

```
┌──────────────────────────────────────────────────────────────────────┐
│                    PRODUCTION ARITHMETIC AGENT                       │
│                                                                      │
│  User Input (natural language)                                       │
│       ↓                                                              │
│  ┌─────────┐     ┌──────────┐     ┌────────────┐                   │
│  │  Parse  │────▶│ Validate │◀───▶│   Fix      │ (loop)            │
│  │  (LLM)  │     │          │     │            │                   │
│  └─────────┘     └────┬─────┘     └────────────┘                   │
│                        ↓                                             │
│                  ┌─────────────┐                                    │
│                  │ Approval    │ (HITL — large multiplications)      │
│                  │ Gate        │                                    │
│                  └──────┬──────┘                                    │
│                         ↓                                           │
│                  ┌─────────────┐                                    │
│                  │   Compute   │ (add / multiply / divide)          │
│                  └──────┬──────┘                                    │
│                         ↓                                           │
│              ┌──────────────────┐                                   │
│              │  Self-Critique   │◀── (re-verify loop)               │
│              │  (LLM Verifier)  │                                   │
│              └────────┬─────────┘                                   │
│                       ↓                                             │
│            ┌──────────────────────┐                                 │
│            │   Memory Write       │ (long-term storage)             │
│            └──────────┬───────────┘                                 │
│                       ↓                                             │
│                  ┌──────────┐                                       │
│                  │ Response │ (streamed to user)                    │
│                  └──────────┘                                       │
│                                                                      │
│  Infrastructure:                                                     │
│  • MemorySaver checkpointer (session persistence)                   │
│  • thread_id per conversation (multi-user support)                  │
│  • MessagesState (message history accumulation)                     │
└──────────────────────────────────────────────────────────────────────┘
```

## LangGraph Concepts Used in This Lesson

| Concept | Where |
|---------|-------|
| `StateGraph`, `TypedDict` | Foundation |
| `add_conditional_edges`, router | Routing all branching decisions |
| `Annotated[list, operator.add]` | History accumulation |
| Validation loop (backward edge) | Retry on invalid input |
| `ChatVertexAI`, `SystemMessage` | LLM parse + critique |
| `MemorySaver`, `thread_id` | Session checkpointing |
| `interrupt()`, `Command(resume=...)` | Human-in-the-loop approval |
| `graph.stream()` | Real-time output |
| Self-critique loop | Answer verification |
| `MessagesState` | Message history |

## Suggested Improvements for Real Production
1. **Replace MemorySaver** with `SqliteSaver` or `PostgresSaver`
2. **Add rate limiting** on LLM nodes (exponential backoff)
3. **Add observability**: LangSmith tracing for every graph execution
4. **Replace SQLite memory** with a vector store (Pinecone, Chroma) for semantic recall
5. **Add authentication**: `thread_id = user_id + session_id`
6. **Add timeout** handling for LLM calls
7. **Multi-language support**: add a translation node before parse

## Suggested Real-World Extensions
- Replace arithmetic nodes with: SQL generation, code execution, API calls
- Add a Planning node (Lesson 9 fan-out): solve multiple sub-problems in parallel
- Add a Router node to support 20+ operations (not just 3)
- Connect to LangSmith for full observability dashboard

## Suggested Advanced Experiments
1. **Benchmark**: measure how often the Critic catches genuine LLM errors
2. **Two models**: use `gemini-2.5-pro` as Critic, `gemini-2.5-flash` as Solver
3. **Streaming HITL**: stream the computation to the human BEFORE asking for approval
4. **Graph compilation caching**: profile `compile()` time for complex graphs
5. **Concurrent sessions**: run 10 simultaneous `thread_id`s and observe isolation

## You Have Now Mastered LangGraph

| Level | Lessons | Skills |
|-------|---------|--------|
| Beginner | 1–5 | StateGraph, nodes, edges, conditional routing, loops |
| Intermediate | 6–12 | LLM integration, tools, parallel, subgraphs, multi-agent |
| Advanced | 13–21 | Memory, streaming, checkpointing, HITL, self-critique, production |
